In [1]:
import os
import json
import pickle
import sqlite3
import numpy as np
import faiss
import pandas as pd
import re
import json
from difflib import SequenceMatcher, get_close_matches
from sentence_transformers import SentenceTransformer
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

True

# Objetivos


In [ ]:
# PATHS

DB_PATH = os.path.join(r"C:\Users\Enrique Guerra\Documents\Claro Insurance\Proyectos\RAG Database", "ventas.db")

EXAMPLES_PATH = os.path.join(
    r"C:\Users\Enrique Guerra\Documents\Claro Insurance\Proyectos\RAG Database\examples",
    "examples.json"
)

EMBEDDINGS_PATH = os.path.join(
    r"C:\Users\Enrique Guerra\Documents\Claro Insurance\Proyectos\RAG Database\embeddings_data",
    "datos_busqueda.pkl"
)

# MODELOS

modelo_embeddings = SentenceTransformer("all-MiniLM-L6-v2")

client = Groq(api_key=os.getenv("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"))


In [33]:
# Normalizar Texto
def normalizar(texto):
    return texto.lower().strip()

def limpiar_json_llm(texto):

    if texto is None:
        return None

    texto = texto.strip()

    # eliminar markdown
    texto = texto.replace("```json", "")
    texto = texto.replace("```", "")

    # buscar primer bloque json
    match = re.search(r"\{.*\}", texto, re.DOTALL)

    if match:
        texto = match.group()

    try:
        return json.loads(texto)
    except:
        return None

def limpiar_entidades_llm(analisis):

    for k in ["agencia","agente","representante","cliente"]:

        if analisis.get(k) in ["", None]:
            analisis[k] = "ninguno"

    return analisis

# Cargar Embeddings
def cargar_datos_busqueda():
    with open(EMBEDDINGS_PATH, "rb") as f:
        datos = pickle.load(f)
    return datos

# Cargar Ejemplos
def cargar_ejemplos():
    with open(EXAMPLES_PATH, "r", encoding="utf-8") as f:
        ejemplos = json.load(f)
    return ejemplos

# Cargar Entidades
def cargar_entidades(db_path=DB_PATH):

    conn = sqlite3.connect(db_path)
    cur = conn.cursor()

    entidades = {}

    entidades["agencias"] = [row[0] for row in cur.execute("SELECT nombre FROM Agencia").fetchall()]
    entidades["agentes"] = [row[0] for row in cur.execute("SELECT nombre FROM Agente").fetchall()]
    entidades["representantes"] = [row[0] for row in cur.execute("SELECT nombre FROM RepresentanteComercial").fetchall()]
    entidades["clientes"] = [row[0] for row in cur.execute("SELECT nombre FROM Cliente").fetchall()]

    conn.close()

    return entidades

# Prompt de Busqueda
def construir_prompt(pregunta):
    prompt = f"""
Eres un analizador semántico para una base de datos comercial.

Debes identificar:

1. categoria (siempre)
2. sub_categoria
3. 1 entidade específica mencionada (si aplica)

IMPORTANTE:

La categoria se refiere al tipo de información solicitada, incluso si
no se menciona una entidad específica.

Ejemplos:

Pregunta: ¿Cuáles agencias existen?
categoria: agencia
agencia: ninguno

Pregunta: ¿Cuánto vendió la Agencia Central?
categoria: agencia
agencia: agencia central

Pregunta: ¿Cuánto vendió la Oficina Central? 
categoria: agencia
agencia: agencia central

Pregunta: ¿Qué agentes pertenecen al representante Carlos Rodriguez?
categoria: agente
representante: carlos rodriguez

Si no se menciona una entidad específica usa "ninguno".
Cuando el usuario use sinónimos como "oficina", "sucursal" o "local" 
para referirse a una agencia, debes devolver el nombre completo 
con "agencia" en lugar del sinónimo.

CATEGORIAS:
agencia
agente
representante
cliente

SUB_CATEGORIAS:
ventas
master

Responde SOLO JSON.

FORMATO:

{{
"categoria":"",
"sub_categoria":"",
"agencia":"",
"agente":"",
"representante":"",
"cliente":""
}}

PREGUNTA:
{pregunta}
"""
    return prompt

# Analizar Pregunta
def analizar_pregunta(pregunta):
    prompt = construir_prompt(pregunta)

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=200
    )

    salida = response.choices[0].message.content
    #print(f"DEBUG - Respuesta LLM cruda: {salida}")  # ← Agrega esto
    resultado = limpiar_json_llm(salida)
    #print(f"DEBUG - JSON parseado: {resultado}")  # ← Agrega esto
    if resultado is None:
        return None

    resultado = limpiar_entidades_llm(resultado)
    return resultado

# Validar Entidad por Similitud
def validar_entidad_por_similitud(entidad_llm, tipo_entidad, entidades_db, umbral=0.5):
    if entidad_llm in ["ninguno", ""]:
        return "ninguno", None, 0.0

    lista_lower = [e.lower() for e in entidades_db[tipo_entidad]]
    entidad_lower = entidad_llm.lower()

    matches = get_close_matches(entidad_lower, lista_lower, n=1, cutoff=umbral)

    if matches:
        idx = lista_lower.index(matches[0])
        entidad_valida = entidades_db[tipo_entidad][idx]  # ✅ devuelve formato original
        nivel_confianza = SequenceMatcher(None, entidad_lower, matches[0]).ratio()
        return entidad_valida, entidad_valida, nivel_confianza

    # Si el valor exacto está en la BD ignorando mayúsculas
    if entidad_lower in lista_lower:
        idx = lista_lower.index(entidad_lower)
        entidad_valida = entidades_db[tipo_entidad][idx]
        return entidad_valida, entidad_valida, 1.0

    return "ninguno", None, 0.0

# Reemplazar Entidades Pregunta
def reemplazar_entidades_por_bd(pregunta, analisis, entidades_originales):
    """
    Reemplaza en la pregunta original la entidad detectada por el nombre válido en BD.
    """
    pregunta_norm = pregunta
    
    # Tu código original como respaldo
    if analisis["agencia"] not in ["ninguno",""]:
        original = entidades_originales.get("agencia")
        if original:
            # Versión mejorada con ignore case
            patron = re.compile(re.escape(original), re.IGNORECASE)
            pregunta_norm = patron.sub(analisis["agencia"], pregunta_norm)

    if analisis["agente"] not in ["ninguno",""]:
        original = entidades_originales.get("agente")
        if original:
            patron = re.compile(re.escape(original), re.IGNORECASE)
            pregunta_norm = patron.sub(analisis["agente"], pregunta_norm)

    if analisis["representante"] not in ["ninguno",""]:
        original = entidades_originales.get("representante")
        if original:
            patron = re.compile(re.escape(original), re.IGNORECASE)
            pregunta_norm = patron.sub(analisis["representante"], pregunta_norm)

    if analisis["cliente"] not in ["ninguno",""]:
        original = entidades_originales.get("cliente")
        if original:
            patron = re.compile(re.escape(original), re.IGNORECASE)
            pregunta_norm = patron.sub(analisis["cliente"], pregunta_norm)

    return pregunta_norm

# Busqueda Vectorial
def buscar_similitud(pregunta, datos_busqueda, entidad, requiere_filtro):

    #entidad = entidad.capitalize()
    entidad = entidad.title()
    
    if entidad not in datos_busqueda:
        return 0, None

    bloque = datos_busqueda[entidad]

    embeddings = np.array(bloque["embeddings"]).astype("float32")
    ejemplos = bloque["ejemplos"]

    emb_usuario = modelo_embeddings.encode(
        [normalizar(pregunta)],
        normalize_embeddings=True
    ).astype("float32")

    indices_validos = []

    for i, e in enumerate(ejemplos):

        filtro_ejemplo = str(e["requiere_filtro"]).lower() == "true"

        if filtro_ejemplo != requiere_filtro:
            continue

        #if e["variable"].lower() != variable.lower():
        #    continue

        indices_validos.append(i)

    if len(indices_validos) == 0:
        print("⚠️ No hay ejemplos después del filtro")
        return 0, None

    embeddings_filtrados = embeddings[indices_validos]

    index = faiss.IndexFlatIP(embeddings.shape[1])

    #print("Requiere filtro:", requiere_filtro)
    #print("Ejemplos válidos:", len(indices_validos))

    index.add(embeddings_filtrados)

    distancias, indices = index.search(emb_usuario, k=1)

    similitud = float(distancias[0][0])

    idx_real = indices_validos[indices[0][0]]

    ejemplo = ejemplos[idx_real]

    return similitud, ejemplo

# SQL Builder
def construir_sql(sql_template, analisis):

    sql = sql_template

    if "{{agencia}}" in sql:
        sql = sql.replace("{{agencia}}", analisis["agencia"].title())

    if "{{agente}}" in sql:
        sql = sql.replace("{{agente}}", analisis["agente"].title())

    if "{{representante}}" in sql:
        sql = sql.replace("{{representante}}", analisis["representante"].title())

    if "{{cliente}}" in sql:
        sql = sql.replace("{{cliente}}", analisis["cliente"].title())

    return sql

# Ejecutar SQL
def ejecutar_sql(query):

    conn = sqlite3.connect(DB_PATH)

    cur = conn.cursor()

    resultado = cur.execute(query).fetchall()

    conn.close()

    return resultado

# Memoria No Match Questions
def guardar_no_match(pregunta, similitud):

    conn = sqlite3.connect("preguntas_no_match.db")

    cur = conn.cursor()

    cur.execute("""
    CREATE TABLE IF NOT EXISTS preguntas (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        pregunta TEXT,
        similitud REAL
    )
    """)

    cur.execute(
        "INSERT INTO preguntas (pregunta, similitud) VALUES (?, ?)",
        (pregunta, similitud)
    )

    conn.commit()

    conn.close()

# Motor Principal
def responder(pregunta_usuario):
    print("\nAnalizando intención...\n")
    
    analisis = analizar_pregunta(pregunta_usuario)

    entidades_db = cargar_entidades()

    entidades_originales = analisis.copy()

    # Validar cada entidad con similitud
    match_agencia, _, conf_agencia = validar_entidad_por_similitud(analisis["agencia"], "agencias", entidades_db)
    analisis["agencia"] = match_agencia  # Actualizamos con el match encontrado
    
    match_agente, _, conf_agente = validar_entidad_por_similitud(analisis["agente"], "agentes", entidades_db)
    analisis["agente"] = match_agente
    
    match_representante, _, conf_representante = validar_entidad_por_similitud(analisis["representante"], "representantes", entidades_db)
    analisis["representante"] = match_representante
    
    match_cliente, _, conf_cliente = validar_entidad_por_similitud(analisis["cliente"], "clientes", entidades_db)
    analisis["cliente"] = match_cliente

    if analisis is None:
        return {"error": "fallo analisis llm"}

    pregunta_normalizada = reemplazar_entidades_por_bd(
        pregunta_usuario,
        analisis,
        entidades_originales
    )

    datos_busqueda = cargar_datos_busqueda()

    requiere_filtro = (
        analisis["agencia"] != "ninguno"
        or analisis["agente"] != "ninguno"
        or analisis["representante"] != "ninguno"
        or analisis["cliente"] != "ninguno"
    )

    sql_final = None
    resultado = None

    #---- Momentaneo para Arreglar el Bug de Representante Comercial
    #---- La solución principal es arreglar "entidad" en examples.json

    MAP_CATEGORIAS = {
        "agencia": "Agencia",
        "agente": "Agente",
        "representante": "Representante Comercial",
        "cliente": "Cliente",
        "producto": "Producto"
    }

    categoria_embeddings = MAP_CATEGORIAS.get(analisis["categoria"])

    similitud, ejemplo = buscar_similitud(
        pregunta_normalizada,
        datos_busqueda,
        categoria_embeddings,
        requiere_filtro
    )

    """
    similitud, ejemplo = buscar_similitud(
        pregunta_normalizada,
        datos_busqueda,
        analisis["categoria"],
        requiere_filtro
    )
    """

    # Crear diccionario base con todos los campos
    respuesta_base = {
        "Pregunta del Usuario": pregunta_usuario,
        "Categoria Detectada": analisis.get("categoria"),
        "Agencia Detectada": analisis.get("agencia"),
        "Agente Detectada": analisis.get("agente"),
        "Representante Detectado": analisis.get("representante"),
        "Cliente Detectado": analisis.get("cliente"),
        "Pregunta Normalizada": pregunta_normalizada,
        "Requiere Filtro": requiere_filtro,
        "Similitud Pregunta": similitud,
        "Match Agencia": match_agencia if 'match_agencia' in locals() else "ninguno",
        "Confianza Agencia": conf_agencia if 'conf_agencia' in locals() else 0.0,
        "Match Agente": match_agente if 'match_agente' in locals() else "ninguno",
        "Confianza Agente": conf_agente if 'conf_agente' in locals() else 0.0,
        "Match Representante": match_representante if 'match_representante' in locals() else "ninguno",
        "Confianza Representante": conf_representante if 'conf_representante' in locals() else 0.0,
        "Match Cliente": match_cliente if 'match_cliente' in locals() else "ninguno",
        "Confianza Cliente": conf_cliente if 'conf_cliente' in locals() else 0.0,
        "SQL": sql_final,
        "SQL Respuesta": resultado
    }

    if similitud < 0.80 or ejemplo is None:
        guardar_no_match(pregunta_usuario, similitud)
        return respuesta_base

    sql_final = construir_sql(ejemplo["sql"], analisis)
    resultado = ejecutar_sql(sql_final)
    
    respuesta_base["SQL"] = sql_final
    respuesta_base["SQL Respuesta"] = resultado
    
    return respuesta_base

# Respuesta Formateada
def respuesta_formateada(respuesta):
    print("\n===== RESULTADO DEL ANÁLISIS =====\n")
    print(f"Pregunta del Usuario   : {respuesta.get('Pregunta del Usuario')}")
    print(f"Categoría Detectada    : {respuesta.get('Categoria Detectada')}")
    print(f"Agencia Detectada      : {respuesta.get('Agencia Detectada')}")
    print(f"Agente Detectada       : {respuesta.get('Agente Detectada')}")
    print(f"Representante Detectado: {respuesta.get('Representante Detectado')}")
    print(f"Cliente Detectado      : {respuesta.get('Cliente Detectado')}")

    print(f"Requiere Filtro        : {respuesta.get('Requiere Filtro')}")
    print(f"Pregunta Normalizada   : {respuesta.get('Pregunta Normalizada')}")
    
    print(f"Match Agencia          : {respuesta.get('Match Agencia')} (Confianza: {(respuesta.get('Confianza Agencia') or 0.0):.2f})")
    print(f"Match Agente           : {respuesta.get('Match Agente')} (Confianza: {(respuesta.get('Confianza Agente') or 0.0):.2f})")
    print(f"Match Representante    : {respuesta.get('Match Representante')} (Confianza: {(respuesta.get('Confianza Representante') or 0.0):.2f})")
    print(f"Match Cliente          : {respuesta.get('Match Cliente')} (Confianza: {(respuesta.get('Confianza Cliente') or 0.0):.2f})")
    
    print(f"Similitud Pregunta     : {respuesta.get('Similitud Pregunta')}\n")
    print(f"SQL                    : {respuesta.get('SQL')}\n")
    
    print("SQL Respuesta          :")
    sql_res = respuesta.get('SQL Respuesta')
    if sql_res:
        for row in sql_res:
            print("  -", row[0] if isinstance(row, tuple) else row)
    else:
        print("  No se encontró respuesta")
    print("\n==================================\n")


In [31]:
def cargar_datos_busqueda():
    with open(EMBEDDINGS_PATH, "rb") as f:
        datos = pickle.load(f)
    return datos

datos_busqueda = cargar_datos_busqueda()
print("Keys embeddings:", datos_busqueda.keys())

Keys embeddings: dict_keys(['Agencia', 'Agente', 'Representante Comercial', 'Cliente', 'Producto'])


In [35]:
pregunta = "¿Cuánto ha vendido un representante comercial específico Carlos Rodríguez?"

respuesta = responder(pregunta)

respuesta_formateada(respuesta) 



Analizando intención...


===== RESULTADO DEL ANÁLISIS =====

Pregunta del Usuario   : ¿Cuánto ha vendido un representante comercial específico Carlos Rodríguez?
Categoría Detectada    : representante
Agencia Detectada      : ninguno
Agente Detectada       : ninguno
Representante Detectado: Carlos Rodríguez
Cliente Detectado      : ninguno
Requiere Filtro        : True
Pregunta Normalizada   : ¿Cuánto ha vendido un representante comercial específico Carlos Rodríguez?
Match Agencia          : ninguno (Confianza: 0.00)
Match Agente           : ninguno (Confianza: 0.00)
Match Representante    : Carlos Rodríguez (Confianza: 1.00)
Match Cliente          : ninguno (Confianza: 0.00)
Similitud Pregunta     : 0.8555352091789246

SQL                    : SELECT r.nombre, SUM(v.monto_total) as ventas_totales FROM RepresentanteComercial r JOIN Agente a ON r.representanteID = a.representanteID JOIN Venta v ON a.agenteID = v.agenteID WHERE r.nombre = 'Carlos Rodríguez' GROUP BY r.nombre;

SQL Respu

In [34]:
datos = cargar_datos_busqueda()

print(type(datos))
print(datos.keys())

<class 'dict'>
dict_keys(['Agencia', 'Agente', 'Representante Comercial', 'Cliente', 'Producto'])


In [30]:
# ---------- FUNCIONES AUXILIARES ---------- 

BASE_DIR = r"C:\Users\Enrique Guerra\Documents\Claro Insurance\Proyectos\RAG Database"
db_path = os.path.join(BASE_DIR, "ventas.db")

conn = sqlite3.connect(db_path)

df = pd.read_sql("SELECT nombre FROM Agencia", conn)
print(df)

            nombre
0  Agencia Central
1    Agencia Norte
2      Agencia Sur
3     Agencia Este
4    Agencia Oeste


In [77]:
BASE_DIR = r"C:\Users\Enrique Guerra\Documents\Claro Insurance\Proyectos\RAG Database"
db_path = os.path.join(BASE_DIR, "ventas.db")

conn = sqlite3.connect(db_path)

df = pd.read_sql("SELECT * FROM RepresentanteComercial", conn)
print(df)

   representanteID            nombre                         email  telefono  \
0                1  Carlos Rodríguez  carlos.rodriguez@empresa.com  555-1001   
1                2    María González    maria.gonzalez@empresa.com  555-1002   
2                3     Juan Martínez     juan.martinez@empresa.com  555-1003   
3                4         Ana López         ana.lopez@empresa.com  555-1004   
4                5     Pedro Sánchez     pedro.sanchez@empresa.com  555-1005   

  fecha_contratacion  
0         2020-01-15  
1         2019-03-20  
2         2021-06-10  
3         2018-11-05  
4         2022-02-28  


In [11]:
import json
from groq import Groq

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def extraer_entidad_llm(pregunta):

    prompt = f"""
Extrae la entidad mencionada en la pregunta.

Tipos posibles:
Agencia
Agente
Representante
Cliente
Ninguno

Responde SOLO JSON.

Formato:

{{
"tipo_entidad":"",
"entidad":""
}}

Pregunta:
{pregunta}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role":"user","content":prompt}],
        temperature=0
    )

    salida = response.choices[0].message.content

    resultado = limpiar_json_llm(salida)
    return resultado

def cargar_entidades(tipo):

    conn = sqlite3.connect(DB_PATH)

    query_map = {
        "Agencia": "SELECT nombre FROM Agencia",
        "Agente": "SELECT nombre FROM Agente",
        "Representante": "SELECT nombre FROM RepresentanteComercial",
        "Cliente": "SELECT nombre FROM Cliente"
    }

    df = pd.read_sql(query_map[tipo], conn)

    conn.close()

    return df["nombre"].tolist()

def crear_indice(lista):

    embeddings = modelo_embeddings.encode(
        lista,
        normalize_embeddings=True
    ).astype("float32")

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)

    return index

def buscar_match(entidad_usuario, lista, index):

    emb = modelo_embeddings.encode(
        [entidad_usuario],
        normalize_embeddings=True
    ).astype("float32")

    dist, idx = index.search(emb, 1)

    match = lista[idx[0][0]]
    similitud = float(dist[0][0])

    return match, similitud

def resolver_pregunta(pregunta):

    print("\nPregunta:", pregunta)

    analisis = extraer_entidad_llm(pregunta)

    tipo = analisis["tipo_entidad"]
    entidad_usuario = analisis["entidad"]

    print("Entidad Detectada:", tipo)
    print("Entidad mencionada:", entidad_usuario)

    if tipo == "Ninguno":
        return

    lista = cargar_entidades(tipo)

    index = crear_indice(lista)

    match, similitud = buscar_match(
        entidad_usuario,
        lista,
        index
    )

    print("Entidad Match:", match)
    print("Similitud:", similitud)

    

In [15]:
pregunta = "Cuales son las ventas del representante comercial Carlos Rodriguez"

resolver_pregunta(pregunta)


Pregunta: Cuales son las ventas del representante comercial Carlos Rodriguez
Entidad Detectada: Representante
Entidad mencionada: Carlos Rodriguez
Entidad Match: Carlos Rodríguez
Similitud: 1.0
